# Shaping Sound: Turning Frequencies Up and Down

In notebook **00** you saw that any sound is a mix of frequencies — each one with its own
loudness. You heard the difference between a flat mix (white noise) and an uneven mix
(bass-heavy coloured noise).

In this notebook we go further. We will:

- Understand **power** — the measure of how loud each frequency is
- **Turn individual frequencies up or down** and hear the result
- **Level out** an uneven mix so every frequency is equally loud
- Use that technique to pull hidden sounds out of noise that completely buries them

Put headphones in. The demos are the whole point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import welch
from IPython.display import Audio, display

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110})

FS       = 22050
DURATION = 5.0
N        = int(FS * DURATION)
t        = np.linspace(0, DURATION, N, endpoint=False)

def play(x, label=''):
    if label:
        print(label)
    normed = (x / (np.max(np.abs(x)) + 1e-10) * 0.9).astype(np.float32)
    display(Audio(normed, rate=FS))

def fingerprint(ax, x, title, xlim=(0, 5000), nperseg=None, color='C0', log_x=False):
    """Show how loud each frequency is (normalised to peak = 1)."""
    if nperseg is None:
        nperseg = min(len(x), FS)
    f, p = welch(x, fs=FS, nperseg=nperseg)
    p_n  = p / (p.max() + 1e-15)
    ax.fill_between(f, p_n, alpha=0.55, color=color)
    ax.plot(f, p_n, lw=1.2, color=color)
    ax.set_xlim(xlim); ax.set_ylim(0, 1.25)
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Loudness')
    ax.set_title(title); ax.grid(alpha=0.3)
    if log_x:
        ax.set_xscale('log')

def specgram(ax, x, title, f_max=3000, vmin=-80, vmax=0):
    """Sound picture: time on x, frequency on y, brightness = loudness."""
    f, ts, Sxx = signal.spectrogram(x, fs=FS, nperseg=1024, noverlap=768, scaling='spectrum')
    db = 10 * np.log10(np.maximum(Sxx, 1e-12))
    ax.pcolormesh(ts, f, db, vmin=vmin, vmax=vmax, cmap='inferno', shading='gouraud')
    ax.set_ylim(0, f_max); ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)

def boost_freq(x, center_hz, width_hz=250, gain=12.0):
    """Turn up the volume at one frequency region."""
    fd    = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), 1.0 / FS)
    boost = 1.0 + (gain - 1.0) * np.exp(-((freqs - center_hz) / (width_hz / 2.5)) ** 2)
    return np.fft.irfft(fd * boost, n=len(x))

def cut_freq(x, center_hz, width_hz=80, depth=0.97):
    """Turn down the volume at one frequency region."""
    fd    = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), 1.0 / FS)
    cut   = 1.0 - depth * np.exp(-((freqs - center_hz) / (width_hz / 2.5)) ** 2)
    return np.fft.irfft(fd * cut, n=len(x))

def equalise(mixed, noise_ref=None, f_low=40.0):
    """Make every frequency equally loud.
    Measures how loud each frequency is on average, then divides to level them out.
    """
    if noise_ref is None:
        noise_ref = mixed
    f_w, psd = welch(noise_ref, fs=FS, nperseg=FS // 4, noverlap=FS // 8)
    level    = np.sqrt(psd)              # how loud each frequency is
    fd       = np.fft.rfft(mixed)
    freqs    = np.fft.rfftfreq(len(mixed), 1.0 / FS)
    L        = np.interp(freqs, f_w, level)
    L[0]     = L[1]
    fd_eq    = fd / L
    fd_eq[freqs < f_low] = 0.0
    return np.fft.irfft(fd_eq, n=len(mixed))

def synth_vowel(f0, formants, dur):
    """Build a vowel from harmonics boosted near the formant frequencies."""
    t_v = np.linspace(0, dur, int(FS * dur), endpoint=False)
    y   = np.zeros(len(t_v))
    k   = 1
    while True:
        freq = k * f0
        if freq > 5000:
            break
        amp = sum(np.exp(-((freq - fi) / bw) ** 2) for fi, bw in formants)
        y  += amp * np.sin(2 * np.pi * freq * t_v)
        k  += 1
    return y / (np.max(np.abs(y)) + 1e-10)

print('Ready!')

---
## 1. Power and the Frequency Fingerprint

### What is power?

The **power** at a given frequency is how much energy the signal carries there — it is
proportional to the *square* of the amplitude. Double the amplitude → four times the power.

A loud bass note carries high power at low frequencies.
A loud whistle carries high power at high frequencies.

### What is the frequency fingerprint?

In notebook 00 you saw the frequency fingerprint as a bar chart. Here we look at it
more carefully: x-axis is frequency, y-axis is how much power (energy) is at that frequency.

Three examples below:
- **One tone** → one spike at its frequency, nothing elsewhere
- **Three tones** → three spikes, one per tone
- **White noise** → flat — *every* frequency equally loud

In [ ]:
t1 = np.linspace(0, 2.0, 2 * FS, endpoint=False)

sig_single = np.sin(2 * np.pi * 440 * t1)
f_s, p_s   = welch(sig_single, fs=FS, nperseg=FS // 2)

sig_three  = (np.sin(2 * np.pi * 300  * t1) +
              np.sin(2 * np.pi * 1000 * t1) +
              np.sin(2 * np.pi * 3000 * t1))
f_3, p_3   = welch(sig_three, fs=FS, nperseg=FS // 2)

wn_short       = np.random.randn(2 * FS)
f_wns, p_wns   = welch(wn_short, fs=FS, nperseg=FS // 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(f_s[1:], p_s[1:], lw=1.5)
axes[0].set_xlim(0, 5000); axes[0].set_ylim(1e-7, 10)
axes[0].axvline(440, color='red', ls='--', lw=1.5, alpha=0.8, label='440 Hz')
axes[0].set_title('Single tone (440 Hz)\none spike in the fingerprint')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Power')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].semilogy(f_3[1:], p_3[1:], lw=1.5, color='C1')
axes[1].set_xlim(0, 5000); axes[1].set_ylim(1e-7, 10)
for fv, lbl in [(300, '300 Hz'), (1000, '1 000 Hz'), (3000, '3 000 Hz')]:
    axes[1].axvline(fv, color='red', ls='--', lw=1.5, alpha=0.8, label=lbl)
axes[1].set_title('Three tones (300, 1000, 3000 Hz)\nthree spikes')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

med = np.median(p_wns[1:])
axes[2].semilogy(f_wns[1:], p_wns[1:], lw=0.8, color='C2', alpha=0.9)
axes[2].axhline(med, color='red', ls='--', lw=2, label='Mean level')
axes[2].set_xlim(0, FS // 2); axes[2].set_ylim(1e-7, 10)
axes[2].set_title('White noise\nflat: every frequency equally loud')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

fig.suptitle('Frequency fingerprint: the power at every frequency', fontsize=12)
plt.tight_layout(); plt.show()

---
## 2. White Noise — Equal Power at Every Frequency

**White noise** has a *flat* frequency fingerprint: every frequency carries exactly the
same power. The name comes from white light, which contains all colours of the visible
spectrum equally.

Three ways to see this at once:

- **Time domain**: random jitter with no pattern and no preferred rhythm.
- **Frequency fingerprint**: a flat horizontal line — no frequency is louder than any other.
- **Sound picture**: uniformly bright at every frequency and every moment — no structure.

White noise is the most featureless possible sound. It sounds like a steady hiss with
no tone, no rhythm, and no structure — the audio equivalent of a blank page.

In [ ]:
white_noise  = np.random.randn(N) * 0.3
f_wn, psd_wn = welch(white_noise, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t[:3000], white_noise[:3000], lw=0.5)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time domain\nRandom jitter — no pattern, no rhythm')

mean_db = 10 * np.log10(np.median(psd_wn[1:]))
axes[1].semilogx(f_wn[1:], 10 * np.log10(psd_wn[1:]), lw=1.2)
axes[1].axhline(mean_db, color='red', ls='--', lw=2, label='Mean level')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Power (dB)')
axes[1].set_title('Frequency fingerprint\nFlat — all frequencies equal')
axes[1].set_xlim(20, FS // 2); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

f_sp, ts_sp, Sxx = signal.spectrogram(white_noise, fs=FS, nperseg=512, noverlap=384)
db_sp = 10 * np.log10(np.maximum(Sxx, 1e-12))
axes[2].pcolormesh(ts_sp, f_sp, db_sp, vmin=-50, vmax=10, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 5000)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('Sound picture\nUniformly bright — no frequency dominates')

fig.suptitle('White noise: equal power at every frequency and every moment', fontsize=12)
plt.tight_layout(); plt.show()

play(white_noise, 'White noise — uniform hiss, no tonal character:')

---
## 3. Coloured Noise — Some Frequencies Much Louder

The opposite of white noise is **coloured noise**: some frequencies carry far more power
than others. The most common type has low frequencies much louder than high ones — giving
it a deep, bass-heavy rumble.

This shape shows up everywhere in the real world: ocean waves, wind, distant traffic,
mechanical vibrations. Low-frequency energy is typically large; high-frequency energy is
small.

We make a version of it by taking white noise and multiplying the low-frequency components
by a large number — boosting the bass until it dominates everything else.

The comparison plot is the key picture: the **gap** between the white and coloured lines
shows where the coloured noise is much louder.

In [ ]:
raw         = np.random.randn(N)
raw_fd      = np.fft.rfft(raw)
freqs_fd    = np.fft.rfftfreq(N, 1.0 / FS)
freqs_fd[0] = 1.0

colour_filt = 1.0 / (np.maximum(freqs_fd, 1.0) / 60.0) ** 2
col_noise   = np.fft.irfft(raw_fd * colour_filt, n=N)
col_noise  /= np.std(col_noise)
col_noise  *= 0.65

f_c, psd_c = welch(col_noise, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scale_w = white_noise / np.std(white_noise)
scale_c = col_noise   / np.std(col_noise)
axes[0].plot(t[:4000], scale_w[:4000], lw=0.6, alpha=0.8, label='White noise')
axes[0].plot(t[:4000], scale_c[:4000], lw=0.7, alpha=0.8, color='C1', label='Coloured noise')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude (normalised)')
axes[0].set_title('Time domain\nColoured noise has large slow swings')
axes[0].legend(fontsize=9)

axes[1].loglog(f_wn[1:], np.sqrt(psd_wn[1:]), lw=2, color='C0', label='White (flat)')
axes[1].loglog(f_c[1:],  np.sqrt(psd_c[1:]),  lw=2, color='C1', label='Coloured (1/f)')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Loudness at each frequency')
axes[1].set_title('Frequency fingerprints compared\nGap = where coloured noise is much louder')
axes[1].legend(fontsize=9); axes[1].grid(True, which='both', alpha=0.3)
axes[1].set_xlim(20, FS // 2)

f_sp2, ts_sp2, Sxx2 = signal.spectrogram(col_noise, fs=FS, nperseg=512, noverlap=384)
db_sp2 = 10 * np.log10(np.maximum(Sxx2, 1e-12))
axes[2].pcolormesh(ts_sp2, f_sp2, db_sp2, vmin=-30, vmax=30, cmap='inferno', shading='gouraud')
axes[2].set_ylim(0, 5000)
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_title('Sound picture\nBright at bottom, dark at top')

fig.suptitle('Coloured noise: low frequencies dominate', fontsize=12)
plt.tight_layout(); plt.show()

play(col_noise, 'Coloured noise — heavy bass rumble:')

---
## 4. Turning a Frequency Up

Now let's do something deliberate. White noise is flat — every frequency is equally loud.

What if we told a computer: *"at 440 Hz, multiply the volume by twelve"*?

The frequency fingerprint grows a bump at 440 Hz. And if you listen, something remarkable
happens: a clear musical tone appears out of the hiss. You are lifting one frequency above
all the others.

The middle panel shows exactly the instruction we gave — a multiplier curve, where 1× means
*no change* and 12× means *twelve times louder*.

In [ ]:
t3 = np.linspace(0, 3.0, int(FS * 3.0), endpoint=False)
white3 = np.random.randn(len(t3))

boosted = boost_freq(white3, center_hz=440, width_hz=250, gain=12.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fingerprint(axes[0], white3,   'Before\nFlat — all frequencies equal',
            xlim=(0, 2000), nperseg=FS // 8, color='C0')

f_line = np.linspace(10, 2000, 2000)
b_curve = 1.0 + (12.0 - 1.0) * np.exp(-((f_line - 440) / 100.0) ** 2)
axes[1].plot(f_line, b_curve, lw=2.5, color='C2')
axes[1].fill_between(f_line, 1, b_curve, where=b_curve > 1, alpha=0.2, color='C2')
axes[1].axhline(1.0, color='k', ls='--', lw=1, alpha=0.5, label='1× = no change')
axes[1].set_xlim(0, 2000); axes[1].set_ylim(0, 14)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Volume multiplier')
axes[1].set_title('The instruction\n1× = unchanged,   12× = twelve times louder')
axes[1].annotate('440 Hz\nboosted 12×', xy=(440, 12), ha='center',
                 fontsize=9, color='C2', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

fingerprint(axes[2], boosted,  'After\n440 Hz stands above everything else',
            xlim=(0, 2000), nperseg=FS // 8, color='C2')
axes[2].axvline(440, color='red', ls='--', lw=1.5, alpha=0.7, label='440 Hz')
axes[2].legend(fontsize=9)

fig.suptitle('Boosting one frequency: a tone emerges from the hiss', fontsize=12)
plt.tight_layout(); plt.show()

play(white3,  'Before — flat hiss:')
play(boosted, 'After — 440 Hz tone stands out:')

---
## 5. Turning a Frequency Down

The same idea works in reverse. Instead of multiplying by a large number, we multiply
by something very close to zero — nearly silencing one frequency region.

Below, we take a **C-major chord** (the notes C, E, and G sounding together) and remove
the E note by turning its frequency almost to zero.

Before: three notes. After: only two remain.

Listen carefully — which note disappears?

In [ ]:
chord = (np.sin(2 * np.pi * 261.63 * t3) +   # C4
          np.sin(2 * np.pi * 329.63 * t3) +   # E4  ← this gets removed
          np.sin(2 * np.pi * 392.00 * t3))     # G4

chord_no_e = cut_freq(chord, center_hz=330, width_hz=60, depth=0.98)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fingerprint(axes[0], chord,     'Before\nThree notes: C (262), E (330), G (392)',
            xlim=(0, 800), nperseg=FS, color='C4')
for fv, lbl, col in [(261.63, 'C', 'green'), (329.63, 'E', 'red'), (392.00, 'G', 'green')]:
    axes[0].axvline(fv, color=col, ls='--', lw=1.3, alpha=0.8, label=lbl)
axes[0].legend(fontsize=9)

f_line2 = np.linspace(10, 800, 800)
c_curve = 1.0 - 0.98 * np.exp(-((f_line2 - 330) / 24.0) ** 2)
axes[1].plot(f_line2, c_curve, lw=2.5, color='C3')
axes[1].fill_between(f_line2, c_curve, 1, where=c_curve < 1, alpha=0.2, color='C3')
axes[1].axhline(1.0, color='k', ls='--', lw=1, alpha=0.5, label='1× = no change')
axes[1].set_xlim(0, 800); axes[1].set_ylim(-0.15, 1.2)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Volume multiplier')
axes[1].set_title('The instruction\n0 = silent,   1 = unchanged')
axes[1].annotate('330 Hz (E)\nnearly silent', xy=(330, 0.02), ha='center',
                 fontsize=9, color='C3', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

fingerprint(axes[2], chord_no_e, 'After\nE removed — C and G remain',
            xlim=(0, 800), nperseg=FS, color='C4')
for fv, lbl in [(261.63, 'C'), (392.00, 'G')]:
    axes[2].axvline(fv, color='green', ls='--', lw=1.3, alpha=0.8, label=f'{lbl} (kept)')
axes[2].axvline(329.63, color='red', ls='--', lw=1.3, alpha=0.7, label='E (removed)')
axes[2].legend(fontsize=8)

fig.suptitle('Cutting one frequency: one note disappears from the chord', fontsize=12)
plt.tight_layout(); plt.show()

play(chord,      'Before — C + E + G:')
play(chord_no_e, 'After — E removed (C + G only):')

---
## 6. Making Everything Equal — The Equaliser

We have boosted one frequency and cut one frequency.
Now, what if we applied the same idea to **every frequency at once**, with the goal of
making them all equally loud?

The steps are simple:

1. For each frequency, measure how loud it currently is.
2. Divide each frequency by its current loudness.

Where it is **loud**: dividing by a large number makes it smaller.
Where it is **quiet**: dividing by a small number makes it relatively larger.

The result: everything ends up at the same level. The sloped fingerprint becomes flat.

$$\text{equalised}(f) = \frac{\text{original}(f)}{\text{loudness at } f}$$

This is **not a trick** — no information is thrown away. You are redistributing what is
already there so that no single frequency dominates. The three steps below show how.

In [ ]:
f_raw, psd_raw = welch(col_noise, fs=FS, nperseg=FS // 4)
level_per_f   = np.sqrt(psd_raw)   # loudness at each frequency

fd_col     = np.fft.rfft(col_noise)
freqs_out  = np.fft.rfftfreq(N, 1.0 / FS)
lev_interp = np.interp(freqs_out, f_raw, level_per_f)
lev_interp[0] = lev_interp[1]

fd_eq                  = fd_col / lev_interp
fd_eq[freqs_out < 20]  = 0.0

col_equalised = np.fft.irfft(fd_eq, n=N)
f_eq, psd_eq  = welch(col_equalised, fs=FS, nperseg=FS // 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].loglog(f_raw[1:], np.sqrt(psd_raw[1:]), lw=2, color='C1')
axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Loudness at each frequency')
axes[0].set_title('Step 1: Coloured noise\nLoud at low f, quiet at high f')
axes[0].set_xlim(20, FS // 2); axes[0].grid(True, which='both', alpha=0.3)

axes[1].loglog(f_raw[1:], level_per_f[1:], lw=2, color='C3')
axes[1].fill_between(f_raw[1:], level_per_f[1:].min() * 0.5, level_per_f[1:],
                     alpha=0.15, color='C3')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Loudness (measured)')
axes[1].set_title('Step 2: Measure loudness at each frequency\nDivide every bin by this curve')
axes[1].set_xlim(20, FS // 2); axes[1].grid(True, which='both', alpha=0.3)

mean_eq = 10 * np.log10(np.median(psd_eq[1:]))
axes[2].semilogx(f_eq[1:], 10 * np.log10(psd_eq[1:]), lw=1.5, color='C2')
axes[2].axhline(mean_eq, color='red', ls='--', lw=2, label='Mean level (now flat)')
axes[2].set_xlabel('Frequency (Hz)'); axes[2].set_ylabel('Power (dB)')
axes[2].set_title('Step 3: After equalising\nFlat — every frequency equally loud')
axes[2].set_xlim(20, FS // 2); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

fig.suptitle('How equalising works: divide each frequency by its own loudness', fontsize=12)
plt.tight_layout(); plt.show()

---
## Demos: Hearing It in Action

Now let's use equalising on sounds you already know from notebook 00.

In each demo:
1. Listen to the signal alone.
2. Listen to it buried in coloured noise — try to hear it (probably can't).
3. Listen after equalising — the signal returns.

In [ ]:
# The equalise function is already defined in the setup cell above.
# Here is a reminder of what it does, in plain terms:
#
#   equalise(mixed, noise_ref)
#
#   1. Measure how loud each frequency is in `noise_ref` (the noise alone).
#   2. For each frequency in `mixed`, divide by that measured loudness.
#   3. The result: every frequency ends up equally loud.
#
print('equalise() is ready to use.')

---
### Demo 1 — A Melody Buried in Noise

A pentatonic melody (C–E–G–A–C) is hidden under coloured noise at a **6:1 amplitude ratio**.

In the sound picture, each note appears as a **horizontal band** at a fixed frequency.
Those bands are invisible in the mixed signal — swamped by the loud low-frequency noise.
After equalising, the noise floor flattens and the melody bands become clearly visible.

In [ ]:
note_freqs = [261.63, 329.63, 392.00, 440.00, 523.25]
note_dur   = DURATION / len(note_freqs)

melody = np.zeros(N)
for i, freq in enumerate(note_freqs):
    start = int(i * note_dur * FS)
    end   = int((i + 1) * note_dur * FS)
    seg_t = np.linspace(0, note_dur, end - start, endpoint=False)
    tone  = (np.sin(2 * np.pi * freq       * seg_t)
           + 0.50 * np.sin(2 * np.pi * 2 * freq * seg_t)
           + 0.25 * np.sin(2 * np.pi * 3 * freq * seg_t)
           + 0.12 * np.sin(2 * np.pi * 4 * freq * seg_t))
    env         = np.ones(len(seg_t))
    fade        = int(0.05 * FS)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    melody[start:end] = tone * env

melody /= np.max(np.abs(melody))
melody *= 0.10

mixed_music    = col_noise + melody
equalised_music = equalise(mixed_music, noise_ref=col_noise)

ratio = np.sqrt(np.mean(col_noise**2)) / np.sqrt(np.mean(melody**2))
print(f'Noise is {ratio:.1f}x louder (RMS) than the melody.\n')
play(melody,          '1) Melody alone:')
play(mixed_music,     '2) Mixed — melody + coloured noise:')
play(equalised_music, '3) After equalising:')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
specgram(axes[0, 0], melody,          'Melody alone\nHorizontal bands = note frequencies',
         f_max=3000, vmin=-90)
specgram(axes[0, 1], col_noise,       'Coloured noise alone\nBright at bottom, dark at top',
         f_max=3000, vmin=-90)
specgram(axes[1, 0], mixed_music,     'Mixed (melody + noise)\nBands invisible — swamped',
         f_max=3000, vmin=-90)
specgram(axes[1, 1], equalised_music, 'After equalising\nBands visible — floor flat',
         f_max=3000, vmin=-50, vmax=10)
for ax in axes.flat:
    ax.set_ylim(50, 3000)
fig.suptitle('Demo 1 — Melody: sound pictures', fontsize=13)
plt.tight_layout(); plt.show()

---
### Demo 2 — A Chirp Buried in Noise

A chirp sweeps from **80 Hz to 1200 Hz** over 5 seconds — a continuously rising tone,
like a bird call going up in pitch (you heard this in notebook 00).

Unlike the melody, the chirp has no fixed note. Its frequency rises continuously.
In the sound picture it appears as a **diagonal line** — time on the x-axis, rising
frequency on the y-axis.

When buried in coloured noise, the diagonal disappears. After equalising, it reappears.

In [ ]:
chirp_sig = signal.chirp(t, f0=80, f1=1200, t1=DURATION, method='quadratic')
ramp      = int(0.05 * FS)
chirp_sig[:ramp]  *= np.linspace(0, 1, ramp)
chirp_sig[-ramp:] *= np.linspace(1, 0, ramp)
chirp_sig *= 0.10

mixed_chirp    = col_noise + chirp_sig
equalised_chirp = equalise(mixed_chirp, noise_ref=col_noise)

ratio_c = np.sqrt(np.mean(col_noise**2)) / np.sqrt(np.mean(chirp_sig**2))
print(f'Noise is {ratio_c:.1f}x louder than the chirp.\n')
play(chirp_sig,      '1) Chirp alone — a rising tone:')
play(mixed_chirp,    '2) Mixed — chirp + coloured noise:')
play(equalised_chirp,'3) After equalising:')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
specgram(axes[0, 0], chirp_sig,       'Chirp alone\nDiagonal = rising frequency',
         f_max=2000, vmin=-90)
specgram(axes[0, 1], col_noise,       'Coloured noise alone',
         f_max=2000, vmin=-90)
specgram(axes[1, 0], mixed_chirp,     'Mixed (chirp + noise)\nDiagonal invisible',
         f_max=2000, vmin=-90)
specgram(axes[1, 1], equalised_chirp, 'After equalising\nDiagonal clearly visible',
         f_max=2000, vmin=-50, vmax=10)
for ax in axes.flat:
    ax.set_ylim(30, 2000)
fig.suptitle('Demo 2 — Chirp: sound pictures', fontsize=13)
plt.tight_layout(); plt.show()

---
### Demo 3 — Speech Buried in Noise

Finally, let's try with voice sounds. Two synthesised vowels — "ahh" and "ee" — are
mixed into the coloured noise at low volume.

The vowels have a distinctive pattern: bright bands at the frequencies boosted by the
mouth shape (remember notebook 00). When buried in coloured noise, those bands vanish
into the bass rumble.

After equalising: the bands reappear. The voice is back.

In [ ]:
ah = synth_vowel(150, formants=[(750, 100), (1100, 130), (2500, 200)], dur=1.5)
ee = synth_vowel(150, formants=[(280, 80),  (2400, 180), (3000, 220)], dur=1.5)

silence = np.zeros(int(FS * 0.4))
speech  = np.concatenate([ah, silence, ee])
N_sp    = len(speech)

# Coloured noise at the same length
raw_sp  = np.random.randn(N_sp)
fd_sp   = np.fft.rfft(raw_sp)
ff_sp   = np.fft.rfftfreq(N_sp, 1.0 / FS); ff_sp[0] = 1.0
col_sp  = np.fft.irfft(fd_sp * (60.0 / np.maximum(ff_sp, 1.0)), n=N_sp)
col_sp /= np.std(col_sp)

speech *= 0.10
mixed_sp     = col_sp + speech
equalised_sp = equalise(mixed_sp, noise_ref=col_sp)

ratio_sp = np.sqrt(np.mean(col_sp**2)) / np.sqrt(np.mean(speech**2))
print(f'Noise is {ratio_sp:.1f}x louder than the speech.\n')
play(speech,       '1) Speech alone — "ahh" then "ee":')
play(mixed_sp,     '2) Mixed — speech + coloured noise:')
play(equalised_sp, '3) After equalising:')

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
specgram(axes[0, 0], speech,        'Speech alone\nBands shift between vowels',
         f_max=4000, vmin=-60)
specgram(axes[0, 1], col_sp,        'Coloured noise alone',
         f_max=4000, vmin=-60)
specgram(axes[1, 0], mixed_sp,      'Mixed — voice invisible',
         f_max=4000, vmin=-60)
specgram(axes[1, 1], equalised_sp,  'After equalising\nVowel bands visible',
         f_max=4000, vmin=-50, vmax=10)
for ax in axes.flat:
    ax.axvline(1.5, color='white', ls=':', lw=1.2, alpha=0.7)
    ax.axvline(1.9, color='white', ls=':', lw=1.2, alpha=0.7)
fig.suptitle('Demo 3 — Speech: sound pictures', fontsize=13)
plt.tight_layout(); plt.show()

---
## Summary

| | White noise | Coloured noise | After equalising |
|---|---|---|---|
| **Frequency fingerprint** | Flat | Steep — loud at low f | Flat |
| **Sounds like** | Uniform hiss | Bass rumble | Uniform hiss |
| **Signal visible?** | — (baseline) | No | **Yes** |

Equalising converts coloured noise *into* white noise. Once the background is flat,
any structured sound hiding in it — a melody, a rising chirp, a voice — stands out
clearly in the sound picture.

The melody has discrete horizontal bands. The chirp has a rising diagonal.
The voice has bands that shift as the vowel changes.
These are all different kinds of **structure**, and equalising is what makes them visible.

---

### What's next?

The next notebook introduces a real situation where this technique is absolutely essential —
where the noise is so severe that nothing else works. But that is a story for the next one.

For now: you have gone from *listening* to sound to *shaping* it. That is exactly the
mindset you need.